In [3]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
from scipy import stats
from scipy.stats import shapiro, levene, f_oneway, kruskal, mannwhitneyu
from scipy.stats import f_oneway
from itertools import combinations
import seaborn as sns
from factor_analyzer import FactorAnalyzer
import glob
import pingouin as pg
from scipy.stats import ttest_ind 
import pyxdf


In [8]:
def get_gaze_data(filepath = "/home/mitchell/Documents/Projects/P8-Project/Dataprocessing Pipeline/Test_Data/ses-13/sub-13_ses-13_task-Baseline/_.xdf"):
    try:
        streams, header = pyxdf.load_xdf(filepath)
    except Exception as e:
        print(f"Error loading XDF file {filepath}: {e}")
        return None, None
    gaze_stream = next((s for s in streams if s["info"]["name"][0] == "GazePointStream"), None)
    if gaze_stream:
        gaze_data = gaze_stream['time_series']
        gaze_ts = gaze_stream['time_stamps']
        return gaze_data, gaze_ts
    else:
        print(f"Warning: Gaze stream not found in {filepath}.")
        return None, None

gaze_data, gaze_ts = get_gaze_data()
# convert to DataFrame with time stamps in one axis and the object being looked at in the other axis
#gaze_df = pd.DataFrame({'time': gaze_ts, 'layer': [l[0] for l in gaze_data]})
# print each unique layer
#print(gaze_df["layer"].unique())

Error loading XDF file /home/mitchell/Documents/Projects/P8-Project/Dataprocessing Pipeline/Test_Data/ses-13/sub-13_ses-13_task-Baseline/_.xdf: file \home\mitchell\Documents\Projects\P8-Project\Dataprocessing Pipeline\Test_Data\ses-13\sub-13_ses-13_task-Baseline\_.xdf does not exist.


In [9]:
#DrinkPerformanceMetrics
def get_performance_metrics(filepath):
    try:
        streams, header = pyxdf.load_xdf(filepath)
    except Exception as e:
        print(f"Error loading XDF file {filepath}: {e}")
        return None, None
    performance_stream = next((s for s in streams if s["info"]["name"][0] == "DrinkPerformanceMetrics"), None)
    if performance_stream:
        performance_data = performance_stream['time_series']
        performance_ts = performance_stream['time_stamps']
        return performance_data, performance_ts
    else:
        print(f"Warning: Performance stream not found in {filepath}.")
        return None, None

In [ ]:

metrics_list = [
    "ideal_ingredient_count",
    "actual_ingredient_count",
    "wrong_ingredients_count",
    "overpour",
    "underpour",
    "pour_deviation",
    "total_score",
    "correct_glass",
    "mishandled_ingredient_count",
    "sum_ideal_amount",
    "sum_actual_amount",
    "time_taken"
]
def get_performance_metrics_dataframe(file_path):
    session_performance_Df = pd.DataFrame()  # Initialize an empty DataFrame to store global performance data
    performance_data, performance_ts = get_performance_metrics(file_path)
    if performance_data is not None and performance_ts is not None and performance_data.size > 0 and performance_ts.size > 0:
        print(f"Processing file: {file_path}")
        if performance_data.size > 0:
            performance_df = pd.DataFrame(performance_data, columns=metrics_list)
            participant_id = int(file_path.split("\\")[1].split("-")[1])  # Extract participant ID from the filepath
            performance_df['participant_id'] = participant_id  # Add participant ID to the dataframe
            # Determine the group based on the task type in the file path
            task_type = file_path.split("_task-")[1].split("\\")[0]
            print(f"Task type: {task_type}")
            if "HighFi" in task_type:
                group = "C"
            elif "MidFi" in task_type:
                group = "B"
            elif "LowFi" in task_type:
                group = "A"
            else:
                group = "Unknown"
            print(f"Group: {group}")
            performance_df['Group'] = group 
            session_performance_Df = performance_df.copy()
    return session_performance_Df


In [11]:
xdf_folder_path = "Test_Data1\\"
for root, dirs, files in os.walk(xdf_folder_path):
    for subdir in dirs:
        if "-Baseline" not in subdir:
            subfolder_path = os.path.join(root, subdir)
            xdf_files = glob.glob(os.path.join(subfolder_path, "*.xdf"))
            for xdf_file in xdf_files:
                performance_data = get_performance_metrics_dataframe(xdf_file)



Processing file: Test_Data1\ses-02\sub-2_ses-2_task-HighFi\sub-2_ses-1_task-HighFi_.xdf
Task type: HighFi
Group: C
Processing file: Test_Data1\ses-04\sub-4_ses-4_task-HighFi\sub-4_ses-1_task-HighFi_.xdf
Task type: HighFi
Group: C


Stream 1: Segments and clock-segments differ


Processing file: Test_Data1\ses-07\sub-7_ses-7_task-LowFi\_.xdf
Task type: LowFi
Group: Unknown
Processing file: Test_Data1\ses-09\sub-9_ses-9_task-MediumFi\_.xdf
Task type: MediumFi
Group: Unknown


Stream 2: Segments and clock-segments differ


Processing file: Test_Data1\ses-10\sub-10_ses-10_task-MediumFi\_.xdf
Task type: MediumFi
Group: Unknown
Processing file: Test_Data1\ses-11\sub-11_ses-11_task-LowFi\_.xdf
Task type: LowFi
Group: Unknown
Processing file: Test_Data1\ses-12\sub-12_ses-12_task-MediumFi\_.xdf
Task type: MediumFi
Group: Unknown
Processing file: Test_Data1\ses-17\sub-17_ses-1_task-LowFi\_.xdf
Task type: LowFi
Group: Unknown
Processing file: Test_Data1\ses-18\sub-18_ses-1_task-HighFi\_.xdf
Task type: HighFi
Group: C


Stream 1: Segments and clock-segments differ


Processing file: Test_Data1\ses-21\sub-21_ses-1_task-HighFi\_.xdf
Task type: HighFi
Group: C
Processing file: Test_Data1\ses-23\sub-23_ses-23_task-HighFi\_.xdf
Task type: HighFi
Group: C


Stream 1: Segments and clock-segments differ


Processing file: Test_Data1\ses-24\sub-24_ses-24_task-HighFi\_.xdf
Task type: HighFi
Group: C


Stream 4: Segments and clock-segments differ


Processing file: Test_Data1\ses-02\sub-2_ses-2_task-HighFi\sub-2_ses-1_task-HighFi_.xdf
Task type: HighFi
Group: C
Performance data:    ideal_ingredient_count  actual_ingredient_count  wrong_ingredients_count  \
0                     5.0                      2.0                      1.0   
1                     5.0                      4.0                      1.0   
2                     7.0                      3.0                      0.0   

     overpour  underpour  pour_deviation  total_score  correct_glass  \
0    0.000000  12.599808       12.599808     0.000000            1.0   
1  125.199997   0.000000      125.199997     7.775391            0.0   
2   10.200001  15.393333       25.593334    58.947571            1.0   

   mishandled_ingredient_count  sum_ideal_amount  sum_actual_amount  \
0                          1.0             180.0         322.200378   
1                          3.0              32.0         157.400009   
2                          3.0              22.0